# ML S4 · Notebook 00 — The Dataset

| | |
|---|---|
| **Session ID** | ML S4 · Notebook 00 of 04 (setup) |
| **Course position** | Machine Learning — Session 4 of 10 (Classification &amp; the Model Family Map) |
| **Block types** | ⚙️ **Setup** — run this once, then leave it alone. |
| **Prerequisites** | None. This notebook only generates data. |
| **Connects back to** | **ML S3** — this is the same `subscription_churn.csv` you used for the confusion matrix and McNemar work. |
| **Connects forward to** | **Notebooks 01–04 of this session** · **ML S5** (ensembles) · **ML S6** (imbalance and thresholds) |
| **CEP linkage** | This dataset is a structural rehearsal for **Employee Turnover**: binary target, moderate imbalance, a retention decision attached to the output. It is not a substitute — the CEP dataset is fixed. |
| **Run requirements** | `pandas`, `numpy`. Run this first; every other notebook in the session reads the CSV it writes. |
| **Checkpoint file** | `subscription_churn.csv` |


## Why this notebook exists

You met this dataset in Session 3. It is regenerated here so that **this session stands on its own** — if you missed S3, or cleared your working folder, or are opening these notebooks six months from now on a different machine, you do not need to go hunting for a file.

The generator is deterministic. Same seed, same 12,000 rows, every time, on any machine. That property matters more than it sounds: when you and the person next to you get different numbers, the first question is always *"are we even looking at the same data?"* — and here the answer is always yes.

Run the cell below once. It writes `subscription_churn.csv` next to this notebook.


---

## 1. The scenario

A subscription business — think broadband, streaming, SaaS — has 12,000 customers on its books. For each one it holds twelve pieces of information, and it knows whether that customer **churned**: cancelled during the observation window.

The business question is not *"who churned?"* — that is already known and already lost. It is **"who is about to?"**, so that a retention team can act while acting is still possible.

That framing is worth pausing on, because it is the same shape as the Employee Turnover CEP. In both cases:

- the target is binary and the **minority class is the one you care about**,
- the model's output is only useful if somebody *does something* with it,
- and the cost of missing a case is not the same as the cost of a false alarm.

Hold that thought. Session 6 is where it turns into arithmetic.


---

## 2. Generating the data

In [2]:
import numpy as np
import pandas as pd

# --- Generate the churn dataset ---
c = np.random.default_rng(4242)
M = 12_000
contract = c.choice(['Month-to-month', 'One year', 'Two year'], M, p=[.55, .26, .19])
tenure   = np.where(contract == 'Month-to-month', c.integers(1, 40, M), c.integers(6, 73, M))
monthly  = c.normal(70, 26, M).clip(19, 125).round(2)
total    = (monthly * tenure * c.normal(1, .04, M)).round(2)
tickets  = c.poisson(np.where(contract == 'Month-to-month', 2.1, 1.0), M)
usage    = c.normal(240, 95, M).clip(5, 700).round(1)
late     = c.poisson(0.75, M).clip(0, 9)
premium  = c.choice([0, 1], M, p=[.68, .32])
nserv    = c.integers(1, 7, M)
age      = c.integers(19, 79, M)
region   = c.choice(['Northeast', 'South', 'Midwest', 'West'], M, p=[.22, .31, .23, .24])
satis    = np.clip(np.round(c.normal(7.1, 1.9, M) - tickets * 0.42 - late * 0.30), 1, 10)


# The churn 'signal'. Read this as a story about why people cancel.
z = (-2.45
     + (contract == 'Month-to-month') * 1.30      # no lock-in, easy to walk
     - (contract == 'Two year') * 0.85            # locked in, unlikely to leave
     - 0.030 * tenure                             # long-standing customers stay
     + 0.235 * tickets                            # support problems drive people out
     - 0.265 * (satis - 7)                        # satisfaction is protective
     + 0.175 * late                               # payment friction
     + 0.0105 * (monthly - 70)                    # expensive plans churn more
     - 0.42 * premium                             # premium support helps
     - 0.055 * nserv                              # more services = stickier
     + 0.028 * (monthly - 70) * (7 - satis) * (satis < 6)   # INTERACTION (see note below)
     + 0.85 * ((tenure < 7) & (contract == 'Month-to-month'))  # THRESHOLD (see note below)
    )
p = 1 / (1 + np.exp(-(z + c.normal(0, 0.55, M))))   # unexplainable randomness
churned = (c.random(M) < p).astype(int)

churn = pd.DataFrame({
    'tenure_months': tenure, 'contract_type': contract, 'monthly_charges': monthly,
    'total_charges': total, 'num_support_tickets_6m': tickets,
    'avg_monthly_usage_gb': usage, 'late_payments_12m': late,
    'has_premium_support': premium, 'num_services': nserv, 'age': age,
    'region': region, 'satisfaction_score': satis.astype(int), 'churned': churned,
})
churn.to_csv('subscription_churn.csv', index=False)
print(f"Saved subscription_churn.csv — {len(churn)} rows, churn rate {churn.churned.mean():.1%}")

Saved subscription_churn.csv — 12000 rows, churn rate 19.9%


---

## 3. The data dictionary

| Column | Type | Meaning |
|---|---|---|
| `tenure_months` | int | How long the customer has been with the company |
| `contract_type` | categorical | Month-to-month · One year · Two year |
| `monthly_charges` | float | Current monthly bill, USD |
| `total_charges` | float | Lifetime billed to date, USD |
| `num_support_tickets_6m` | int | Support tickets raised in the last six months |
| `avg_monthly_usage_gb` | float | Average monthly data usage |
| `late_payments_12m` | int | Late payments in the last twelve months |
| `has_premium_support` | binary | 1 if on the premium support tier |
| `num_services` | int | How many separate products they subscribe to |
| `age` | int | Customer age in years |
| `region` | categorical | Northeast · South · Midwest · West |
| `satisfaction_score` | int 1–10 | Latest survey score |
| **`churned`** | **binary** | **1 = cancelled. This is the target.** |


In [3]:
churn.head(8)

,tenure_months,contract_type,monthly_charges,total_charges,num_support_tickets_6m,avg_monthly_usage_gb,late_payments_12m,has_premium_support,num_services,age,region,satisfaction_score,churned
0,41,One year,115.84,4884.81,1,464.4,1,0,6,19,Northeast,6,0
1,39,One year,94.16,3660.67,0,220.7,1,0,6,64,South,6,0
2,23,Month-to-month,52.09,1226.30,2,377.9,1,0,3,63,South,7,0
3,26,Month-to-month,50.29,1368.94,1,337.3,0,1,3,34,South,5,0
4,27,Month-to-month,76.45,2139.25,3,182.2,1,0,6,36,Midwest,2,1
5,9,Month-to-month,122.16,1033.11,0,5.0,0,1,5,37,Northeast,7,0
6,1,Month-to-month,30.46,29.21,0,105.5,2,0,6,60,Midwest,9,0
7,20,Month-to-month,62.27,1337.50,1,327.1,0,0,2,66,Midwest,5,1


In [4]:
print(churn.dtypes.to_string())
print()
print("Missing values:", churn.isna().sum().sum())
print("Duplicate rows:", churn.duplicated().sum())
print()
print("Class balance:")
print(churn.churned.value_counts().to_string())
print(f"\nPositive class = {churn.churned.mean():.2%} of rows")
print(f"Always-predict-'no-churn' accuracy = {1 - churn.churned.mean():.2%}   <-- the number to beat")

tenure_months               int64
contract_type                 str
monthly_charges           float64
total_charges             float64
num_support_tickets_6m      int64
avg_monthly_usage_gb      float64
late_payments_12m           int64
has_premium_support         int64
num_services                int64
age                         int64
region                        str
satisfaction_score          int64
churned                     int64

Missing values: 0
Duplicate rows: 0

Class balance:
churned
0    9618
1    2382

Positive class = 19.85% of rows
Always-predict-'no-churn' accuracy = 80.15%   <-- the number to beat


### That last number is the whole session in one line

**80.2% accuracy** is available to a model that does nothing at all — one that ignores every column and answers "no churn" to every customer, forever.

Any classifier you build today has to be measured against that floor, not against zero. A model reporting 82% accuracy has not learned very much. You already know this from Session 3; the dataset is built so that you cannot forget it.


---

## 4. Three things deliberately planted in this data

This is synthetic data, which has one honest cost: it is cleaner and better-behaved than anything you will meet at work. The compensating benefit is that **every trap in it was put there on purpose**, which means every trap is one you can fully understand rather than merely survive.

### 4.1 Three columns carry no signal at all

`avg_monthly_usage_gb`, `age` and `region` never appear in the `z` formula. They are pure noise — realistic-looking, plausible to a stakeholder, and completely useless.

They are there because **real datasets are mostly noise**, and a practitioner who cannot tell a useless feature from a useful one will ship a model that memorises coincidences.

### 4.2 `total_charges` is almost a restatement of two other columns

Look at the generator: `total = monthly × tenure × (a little noise)`. That makes `total_charges` heavily correlated with both `monthly_charges` and `tenure_months`.

This is not an error. It is what real billing tables look like, and it will do something specific and visible to your logistic regression coefficients in Notebook 02.

### 4.3 Two patterns a straight line cannot represent

Two lines in the generator are flagged in the comments, and they are the reason this session has a fair fight in it.

**The interaction term.** Customers on expensive plans who are *also* dissatisfied churn far more than either factor alone would predict. The effect of price **depends on** satisfaction. That is a genuine interaction.

**The threshold effect.** Brand-new month-to-month customers — under seven months — churn at a sharply elevated rate. The jump is abrupt, not gradual.

Why this matters: **logistic regression cannot represent either pattern** unless you explicitly hand it the interaction. It fits one straight-line effect per feature and stops. A tree-based model discovers both on its own, without being told.

So when Notebook 02 puts a linear model and a tree side by side, the difference between them will be real and traceable to a specific line of the generator — not an artefact of tuning luck.


In [5]:
# The three planted facts, confirmed rather than asserted.

print("--- 4.1  The three decoys: correlation with the target ---")
for col in ['avg_monthly_usage_gb', 'age', 'num_support_tickets_6m', 'satisfaction_score']:
    r = churn[col].corr(churn.churned)
    tag = "  <- decoy" if col in ('avg_monthly_usage_gb', 'age') else ""
    print(f"  {col:<26} r = {r:+.4f}{tag}")

print("\n--- 4.2  total_charges is a restatement ---")
print(f"  corr(total_charges, tenure_months)  = {churn.total_charges.corr(churn.tenure_months):.3f}")
print(f"  corr(total_charges, monthly_charges) = {churn.total_charges.corr(churn.monthly_charges):.3f}")

print("\n--- 4.3a  The threshold effect (month-to-month customers only) ---")
mm = churn[churn.contract_type == 'Month-to-month']
print(f"  under 7 months tenure : churn rate {mm[mm.tenure_months < 7].churned.mean():.1%}")
print(f"  7+ months tenure      : churn rate {mm[mm.tenure_months >= 7].churned.mean():.1%}")

print("\n--- 4.3b  The interaction (price x dissatisfaction) ---")
cheap_happy = churn[(churn.monthly_charges < 70) & (churn.satisfaction_score >= 6)]
dear_happy  = churn[(churn.monthly_charges >= 70) & (churn.satisfaction_score >= 6)]
cheap_unhap = churn[(churn.monthly_charges < 70) & (churn.satisfaction_score < 6)]
dear_unhap  = churn[(churn.monthly_charges >= 70) & (churn.satisfaction_score < 6)]
print(f"  cheap + satisfied   : {cheap_happy.churned.mean():.1%}")
print(f"  costly + satisfied  : {dear_happy.churned.mean():.1%}")
print(f"  cheap + unsatisfied : {cheap_unhap.churned.mean():.1%}")
print(f"  costly + unsatisfied: {dear_unhap.churned.mean():.1%}   <- more than the parts predict")

--- 4.1  The three decoys: correlation with the target ---
  avg_monthly_usage_gb       r = -0.0017  <- decoy
  age                        r = -0.0124  <- decoy
  num_support_tickets_6m     r = +0.2597
  satisfaction_score         r = -0.2631

--- 4.2  total_charges is a restatement ---
  corr(total_charges, tenure_months)  = 0.833
  corr(total_charges, monthly_charges) = 0.470

--- 4.3a  The threshold effect (month-to-month customers only) ---
  under 7 months tenure : churn rate 50.5%
  7+ months tenure      : churn rate 27.4%

--- 4.3b  The interaction (price x dissatisfaction) ---
  cheap + satisfied   : 10.8%
  costly + satisfied  : 14.2%
  cheap + unsatisfied : 11.0%
  costly + unsatisfied: 55.4%   <- more than the parts predict


---

## 5. Sanity check: does the signal look like the story?

If the generator says month-to-month customers churn more, the data had better agree. This is a habit worth keeping for real datasets too — before modelling anything, check that the obvious relationships point the way domain knowledge says they should. When they don't, you have either found something interesting or broken something, and it is nearly always the second one.


In [6]:
print("Churn rate by contract type:")
print(churn.groupby('contract_type').churned.mean().sort_values(ascending=False).round(3).to_string())

print("\nChurn rate by satisfaction score:")
print(churn.groupby('satisfaction_score').churned.mean().round(3).to_string())

print("\nChurn rate by premium support:")
print(churn.groupby('has_premium_support').churned.mean().round(3).to_string())

Churn rate by contract type:
contract_type
Month-to-month    0.311
One year          0.072
Two year          0.050

Churn rate by satisfaction score:
satisfaction_score
1     0.419
2     0.468
3     0.372
4     0.345
5     0.284
6     0.176
7     0.136
8     0.090
9     0.077
10    0.056

Churn rate by premium support:
has_premium_support
0    0.212
1    0.171


---

## What you now have

One file sits next to this notebook:

| File | Rows | Target | Positive rate | Used by |
|---|---|---|---|---|
| `subscription_churn.csv` | 12,000 | `churned` (binary) | 19.9% | ML S4 · S5 · S6 |

Everything in this session runs off it. Notebook 03 also borrows `contract_type` as a **three-class** target, so you get a multiclass problem without a second download.

---

**Next:** open `ML_S4_01_recap_and_problem_types.ipynb`.
